<a href="https://colab.research.google.com/github/k2-fsa/colab/blob/master/itn_number_zh.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%%shell

pip install --only-binary :all: pynini

In [ ]:
import pynini
from pynini.lib import utf8, byte
from pynini.lib.pynutil import delete, insert

# 数字格式日期转中文

In [ ]:
# 数字到中文映射
digit_map = [
    ("0", "零"),
    ("1", "一"),
    ("2", "二"),
    ("3", "三"),
    ("4", "四"),
    ("5", "五"),
    ("6", "六"),
    ("7", "七"),
    ("8", "八"),
    ("9", "九"),
]
zero_to_nine = pynini.string_map(digit_map).optimize()
one_to_nine = pynini.string_map(digit_map[1:]).optimize()
two_to_nine = pynini.string_map(digit_map[2:]).optimize()

ten = delete('0').ques + ((delete('1') | two_to_nine) + insert('十')).ques + (one_to_nine | delete('0'))
ten = ten.optimize()

sigma = utf8.VALID_UTF8_CHAR.star
space = delete(" ").ques

# # 日期部分
year1 = (one_to_nine + zero_to_nine ** (2, 3) + space + delete("年") + insert("年")).optimize()
year2 = (one_to_nine + zero_to_nine ** (2, 3) + space + delete("/") + insert("年")).optimize()
year3 = (one_to_nine + zero_to_nine ** (2, 3) + space + delete("-") + insert("年")).optimize()

month1 = (ten + space + delete("月") + insert("月")).optimize()
month2 = (ten + space + delete("/").ques + insert("月")).optimize()
month3 = (ten + space + delete("-").ques + insert("月")).optimize()

day = (ten + space + delete("日").ques + insert("日")).optimize()

ymd1 = (year1 + space + month1 + space + day).optimize()
ymd2 = (year2 + space + month2 + space + day).optimize()
ymd3 = (year3 + space + month3 + space + day).optimize()

ym1 = (year1 + space + month1).optimize()
ym2 = (year2 + space + month2).optimize()
ym3 = (year3 + space + month3).optimize()

date1 = ymd1 | ymd2 | ymd3
date2 = ym1 | ym2 | ym3
date3 = year1

for i in range(date1.num_states()):
    if date1.final(i) == pynini.Weight.one(date1.weight_type()):
        date1.set_final(i, 10)

for i in range(date2.num_states()):
    if date2.final(i) == pynini.Weight.one(date2.weight_type()):
        date2.set_final(i, 20)

for i in range(date3.num_states()):
    if date3.final(i) == pynini.Weight.one(date3.weight_type()):
        date3.set_final(i, 30)

date = (date1 | date2 | date3).optimize()
# 转换规则
rule = pynini.cdrewrite(date, "", "", sigma).optimize()


for text in [
    "李白（701年2月28日—762年12月），字太白", "2024年", "2024年 9月", "2024/09", "2024年09月 01日", "624年11月",
    "624年11月02日", "2024-09～2025-9", "2024年09月01日-2025年", "-2024年"
]:
   out_put = list(pynini.shortestpath(pynini.compose(text, rule)).paths().ostrings())
   print(out_put)


# 保存 FST
rule.write("date.fst")


! ls -lh date.fst

['李白（七零一年二月二十八日—七六二年十二月），字太白']
['二零二四年']
['二零二四年九月']
['二零二四年月九日']
['二零二四年九月一日']
['六二四年十一月']
['六二四年十一月二日']
['二零二四年月九日～二零二五年九月']
['二零二四年九月一日-二零二五年']
['-二零二四年']
-rw-r--r-- 1 root root 64K Apr  9 08:53 date.fst


# 数字格式的时间转中文

In [ ]:
sigma = utf8.VALID_UTF8_CHAR.star
time_flag = "<$><$>"
morning_flag = "<$>上午<$>"
afternoon_flag = "<$>下午<$>"
# 数字到中文映射
digit_map = [
    ("0", "零"),
    ("1", "一"),
    ("2", "二"),
    ("3", "三"),
    ("4", "四"),
    ("5", "五"),
    ("6", "六"),
    ("7", "七"),
    ("8", "八"),
    ("9", "九"),
]
zero_to_nine = pynini.string_map(digit_map[1:]).optimize()
two_to_nine = pynini.string_map(digit_map[2:]).optimize()

ten = ((delete('1') | two_to_nine) + insert('十')).ques + (zero_to_nine | delete('0'))
ten = ten.optimize()

space = delete(" ").ques
colon = delete(":") | delete("：")

hour = pynini.cross("0", "零") ** (0, 1) + pynini.union(*map(str, range(0, 24))) @ ten
minute = pynini.cross("0", "零") ** (0, 1) + pynini.union(*map(str, range(0, 60))) @ ten
second = pynini.cross("0", "零") ** (0, 1) + pynini.union(*map(str, range(0, 60))) @ ten

time1 = hour + space + colon + insert('点') + space + minute + insert(
    '分') + space + colon + space + second + insert('秒')
time2 = hour + space + colon + insert('点') + space + minute + insert('分')

apm = (space + (pynini.cross(pynini.union("am", "AM", "Am", "aM"), morning_flag) | pynini.cross(
    pynini.union("pm", "PM", "Pm", "pM"), afternoon_flag))).ques

for i in range(time1.num_states()):
    if time1.final(i) == pynini.Weight.one(time1.weight_type()):
        time1.set_final(i, 10)

for i in range(time2.num_states()):
    if time2.final(i) == pynini.Weight.one(time2.weight_type()):
        time2.set_final(i, 20)

time = insert(time_flag) + (time1 | time2) + space + apm
time = pynini.cdrewrite(time, "", "", sigma).optimize()

# 使用 rewrite 将 "上午"/"下午" 调整到前面
rewrite1 = pynini.cross(time_flag, "上午") + pynini.closure(sigma) + delete(morning_flag)
rewrite2 = pynini.cross(time_flag, "下午") + pynini.closure(sigma) + delete(afternoon_flag)
rewrite3 = delete(time_flag)

for i in range(rewrite1.num_states()):
    if rewrite1.final(i) == pynini.Weight.one(rewrite1.weight_type()):
        rewrite1.set_final(i, 30)

for i in range(rewrite2.num_states()):
    if rewrite2.final(i) == pynini.Weight.one(rewrite2.weight_type()):
        rewrite2.set_final(i, 40)

for i in range(rewrite3.num_states()):
    if rewrite3.final(i) == pynini.Weight.one(rewrite3.weight_type()):
        rewrite3.set_final(i, 50)

rewrite = pynini.cdrewrite((rewrite1 | rewrite2 | rewrite3), "", "", sigma).optimize()

rule = pynini.compose(time, rewrite).optimize()


for text in [
    "其他内容4:00pm", "其他内容12:00", "其他内容12:00:01", "其他内容01:01:00 am", "其他内容1:01:10 am",
    "其他内容01:01:01 am还好", "其他内容01:01:00 pm", "其他内容00:00:00 am"
]:
   out_put = list(pynini.shortestpath(pynini.compose(text, rule)).paths().ostrings())
   print(out_put)

rule.write("time.fst")

! ls -lh time.fst

['其他内容下午四点零分']
['其他内容十二点零分']
['其他内容十二点零分零一秒']
['其他内容上午零一点零一分零秒']
['其他内容上午一点零一分十秒']
['其他内容上午零一点零一分零一秒还好']
['其他内容下午零一点零一分零秒']
['其他内容上午零点零分零秒']
-rw-r--r-- 1 root root 162K Apr  9 08:53 time.fst


# 口语中常见的2转两

In [ ]:
sigma = utf8.VALID_UTF8_CHAR.star
two = pynini.cross("2", "两")
unit = pynini.union(*['个', '位', '天', '本', '只', '头', '斤', '米', '台', '杯', '辆', '毛钱', '块钱', '分钱',
                      '年', '双', '对', '副', '次', '下', '圈', '排', '层', '升', '公斤', '小时', '分钟'])
space = delete(' ').ques

rewrite_rule = (two + space + unit).optimize()
rewrite_rule = pynini.cdrewrite(rewrite_rule, "", "", sigma).optimize()

digit = pynini.lib.byte.DIGIT.plus
revert_rule = (digit + pynini.cross("两", '2') + unit).optimize()
revert_rule = pynini.cdrewrite(revert_rule, "", "", sigma).optimize()

rule = pynini.compose(rewrite_rule, revert_rule)

for text in [ "为你找到以下2 个日程", "2 个日程", "12 个日程", "2副手套"]:
   out_put = list(pynini.shortestpath(pynini.compose(text, rule)).paths().ostrings())
   print(out_put)


# 保存 FST
rule.write("two.fst")

! ls -lh two.fst

['为你找到以下两个日程']
['两个日程']
['12个日程']
['两副手套']
-rw-r--r-- 1 root root 117K Apr  9 08:53 two.fst


# 常见电话/座机转换

In [ ]:
# 定义数字到中文的映射
zero = pynini.cross("0", "零")
yao = pynini.cross("1", "幺")
two = pynini.cross("2", "二")
three = pynini.cross("3", "三")
four = pynini.cross("4", "四")
five = pynini.cross("5", "五")
six = pynini.cross("6", "六")
seven = pynini.cross("7", "七")
eight = pynini.cross("8", "八")
nine = pynini.cross("9", "九")

digit = (zero | yao | two | three | four | five | six | seven | eight | nine).optimize()

revert_zero = pynini.cross("零", "0")
revert_yao = pynini.cross("幺", "1")
revert_two = pynini.cross("二", "2")
revert_three = pynini.cross("三", "3")
revert_four = pynini.cross("四", "4")
revert_five = pynini.cross("五", "5")
revert_six = pynini.cross("六", "6")
revert_seven = pynini.cross("七", "7")
revert_eight = pynini.cross("八", "8")
revert_nine = pynini.cross("九", "9")
revert_digit = (revert_zero | revert_yao | revert_two | revert_three | revert_four | revert_five | revert_six | revert_seven | revert_eight | revert_nine).optimize()

space = delete(' ').ques

# 输入字符集
sigma = utf8.VALID_UTF8_CHAR.star

def generate(only_digit_rule):
    only_digit_rule = pynini.cdrewrite(only_digit_rule, "", "", sigma).optimize()
    revert_rule1 = (pynini.lib.byte.DIGIT.plus + revert_digit.plus).optimize()
    revert_rule2 = (revert_digit.plus + pynini.lib.byte.DIGIT.plus).optimize()
    revert_rule = pynini.cdrewrite((revert_rule1 | revert_rule2), "", "", sigma).optimize()
    return pynini.compose(only_digit_rule, revert_rule).optimize()

def fix_phone_rule():
    # 固定电话号码列表
    phones = [
        "110", "120", "119", "122", "12121", "96121", "12306", "12110",
        "95119", "114", "160", "117", "11185", "170", "189", "184", "106",
        "108", "115", "12395", "2580", "2585", "221", "999", "12315", "12358",
        "12366", "12333", "12365", "12310", "12369", "95598", "96300", "12319",
        "12345", "12348", "12318", "12388", "12320", "96102", "10000", "10086",
        "10010", "10060", "10050", "17900", "17951", "10010", "95566", "95588",
        "95533", "95599", "95559", "95595", "95580", "95568", "95516", "95508",
        "95501", "95528", "95555", "95577", "8008302880", "8008301880", "8008303811",
        "95561", "95558", "96169", "95519", "95500", "95511", "95589", "95518", "95505",
        "95507", "95509", "95585", "95556", "95590", "95569", "95522", "95567", "65588598"
    ]
    return pynini.compose(pynini.union(*phones), digit.plus).optimize()

def mobile_phone_rule():
    return (yao + (three | four | five | six | seven | eight | nine) + (digit ** 9)).optimize()

def area_code_rule():
    area_codes = [
        "010", "021", "022", "023", "852", "853", "0898", "0311", "0313", "0314", "0335", "0315", "0316", "0312", "0317", "0318",
        "0319", "0310", "0571", "0572", "0573", "0580", "0574", "0575", "0570", "0579", "0576", "0577", "0578", "024", "0421",
        "0418", "0419", "0412", "0415", "0411", "0417", "0427", "0416", "0429", "0731", "0744", "0736", "0737", "0730", "0734",
        "0735", "0746", "0739", "0745", "0738", "0743", "025", "0516", "0518", "0527", "0517", "0515", "0514", "0523", "0513",
        "0511", "0519", "0510", "0512", "0471", "0472", "0473", "0476", "0475", "0470", "0477", "0474", "0478", "0482", "0479",
        "0483", "0791", "0792", "0798", "0701", "0790", "0799", "0797", "0793", "0794", "0795", "0796", "0351", "0352", "0349",
        "0353", "0355", "0356", "0350", "0354", "0357", "0359", "0358", "0931", "0935", "0943", "0938", "0937", "0936", "0934",
        "0933", "0932", "0939", "0930", "0941", "0531", "0635", "0534", "0546", "0533", "0536", "0535", "0631", "0532", "0633",
        "0539", "0632", "0537", "0538", "0634", "0543", "0530", "0451", "0452", "0456", "0459", "0458", "0468", "0454", "0469",
        "0464", "0467", "0453", "0455", "0457", "0591", "0599", "0598", "0594", "0595", "0592", "0596", "0597", "0593", "020",
        "0763", "0751", "0762", "0753", "0768", "0754", "0663", "0660", "0752", "0769", "0755", "0756", "0760", "0750", "0757",
        "0758", "0766", "0662", "0668", "0759", "028", "0839", "0816", "0838", "0817", "0826", "0825", "0832", "0833", "0813",
        "0831", "0812", "0827", "0818", "0835", "0837", "0836", "0834", "0830", "027", "0719", "0710", "0724", "0712", "0713",
        "0711", "0714", "0715", "0716", "0717", "0722", "0728", "0718", "0371", "0398", "0379", "0391", "0372", "0393", "0370",
        "0374", "0395", "0375", "0377", "0376", "0394", "0396", "0871", "0874", "0877", "0875", "0870", "0888", "0879", "0883",
        "0692", "0886", "0887", "0872", "0878", "0873", "0876", "0691", "0551", "0557", "0561", "0558", "0552", "0554", "0550",
        "0555", "0553", "0562", "0556", "0559", "0564", "0566", "0563", "0951", "0952", "0953", "0954", "0955", "0431", "0436",
        "0438", "0432", "0434", "0437", "0435", "0439", "0433", "0771", "0773", "0772", "0774", "0775", "0777", "0779", "0770",
        "0776", "0778", "0851", "0858", "0857", "0856", "0855", "0854", "0859", "029", "0911", "0919", "0913", "0917", "0916",
        "0912", "0915", "0914", "0971", "0972", "0970", "0974", "0973", "0975", "0976", "0977", "0979", "0891", "0896", "0895",
        "0894", "0893", "0892", "0897", "0991", "0990", "0992", "0993", "0994", "0997", "0996", "0998", "0903", "0995", "0902",
        "0908", "0909", "0901"
    ]
    area_code_rules = pynini.union(*[pynini.compose(p, digit.plus) for p in area_codes])
    area_code_rules = ((delete("(") + space + area_code_rules + space + delete(")")) | (area_code_rules + space + delete("-"))).optimize()
    phone_rules = (digit ** 5 | digit ** (7, ...)).optimize()
    return (area_code_rules + space + phone_rules).optimize()

def other_rule():
    return ((eight | four) + zero + zero + ((digit + delete('-') + (digit ** 3) + delete('-') + (digit ** 3)) | (delete('-') + (digit ** 3) + delete('-') + (digit ** 4)))).optimize()

area_code = pynini.cdrewrite(area_code_rule(), "", "", sigma).optimize()
mobile_phone = pynini.cdrewrite(mobile_phone_rule(), "", "", sigma).optimize()
other = pynini.cdrewrite(other_rule(), "", "", sigma).optimize()
fix_phone = generate(fix_phone_rule())

for i in range(area_code.num_states()):
    if area_code.final(i) == pynini.Weight.one(area_code.weight_type()):
        area_code.set_final(i, 10)

for i in range(mobile_phone.num_states()):
    if mobile_phone.final(i) == pynini.Weight.one(mobile_phone.weight_type()):
        mobile_phone.set_final(i, 20)

for i in range(other.num_states()):
    if other.final(i) == pynini.Weight.one(other.weight_type()):
        other.set_final(i, 30)

for i in range(fix_phone.num_states()):
    if fix_phone.final(i) == pynini.Weight.one(fix_phone.weight_type()):
        fix_phone.set_final(i, 50)

# 构建最终 FST
rule = area_code | mobile_phone | other
rule = pynini.compose(rule, fix_phone).optimize()


for text in [
    "110", "拨打110报警", "(010)12345678", "拨打(010)12345678报警", "拨打1100报警",
    '0 0', '1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12', '100', '101', '110', '120', '121',
]:
   out_put = list(pynini.shortestpath(pynini.compose(text, rule)).paths().ostrings())
   print(out_put)


# 写入到文件
rule.write("phone.fst")

! ls -lh phone.fst

['幺幺零']
['拨打幺幺零报警']
['零幺零幺二三四五六七八']
['拨打零幺零幺二三四五六七八报警']
['拨打1100报警']
['0 0']
['1']
['2']
['3']
['4']
['5']
['6']
['7']
['8']
['9']
['10']
['11']
['12']
['100']
['101']
['幺幺零']
['幺二零']
['121']
-rw-r--r-- 1 root root 181M Apr  9 08:55 phone.fst


# 温度转换

In [ ]:
sigma = utf8.VALID_UTF8_CHAR.star
digit = byte.DIGIT.plus
fraction = (pynini.union('.') + digit).optimize().ques

negative = pynini.cross("-", "零下").ques
optional_space = delete(" ").ques
celsius = optional_space + pynini.cross("℃", "摄氏度")
celsius_1 = optional_space + pynini.cross("度", "度")
celsius_2 = optional_space + pynini.cross("°C", "摄氏度")
fahrenheit = optional_space + pynini.cross("℉", "华氏度")
fahrenheit_1 = optional_space + pynini.cross("°F", "华氏度")

# 温度规则组合
temperature = (negative + digit + fraction + (celsius | celsius_1 | celsius_2 | fahrenheit | fahrenheit_1)).optimize()
# 转换规则
rule = pynini.cdrewrite(temperature, "", "", sigma)

for text in [
    "丰台区今天阴,-4度到5度0 ℃,PM2.5浓度26-1.5 ℃,空气质量优-4°C5°F", "-4℃", "0 ℃", "-1.5 ℃", "33.80℉", "-35.60 ℉",
    "100 ℃", "-273.15 ℃"
]:
   out_put = list(pynini.shortestpath(pynini.compose(text, rule)).paths().ostrings())
   print(out_put)

rule.write('temperature.fst')

! ls -lh temperature.fst

['丰台区今天阴,零下4度到5度0摄氏度,PM2.5浓度26零下1.5摄氏度,空气质量优零下4摄氏度5华氏度']
['零下4摄氏度']
['0摄氏度']
['零下1.5摄氏度']
['33.80华氏度']
['零下35.60华氏度']
['100摄氏度']
['零下273.15摄氏度']
-rw-r--r-- 1 root root 35K Apr  9 08:55 temperature.fst


# 标准数字转换

In [ ]:
sigma = utf8.VALID_UTF8_CHAR.star
one_to_nine_map = [
    ("1", "一"),
    ("2", "二"),
    ("3", "三"),
    ("4", "四"),
    ("5", "五"),
    ("6", "六"),
    ("7", "七"),
    ("8", "八"),
    ("9", "九"),
]

zero = pynini.cross("0", "零")
one_to_nine = pynini.string_map(one_to_nine_map).optimize()
two_to_nine = pynini.string_map(one_to_nine_map[1:]).optimize()

ten = (delete('1') | two_to_nine) + insert('十') + (one_to_nine | delete('0'))
ten = ten.optimize()

# X10-X99
suffix_ten = one_to_nine + insert('十') + (one_to_nine | delete('0'))
suffix_ten = suffix_ten.optimize()

# 100-9,99
hundred = one_to_nine + insert('百') + (suffix_ten | (zero + one_to_nine) | delete('0') ** 2)
hundred = hundred.optimize()

# 1000-9,999
thousand = one_to_nine + insert('千') + (hundred | (zero + suffix_ten) | (delete('0') + zero + one_to_nine) | delete('0') ** 3)
thousand = thousand.optimize()

# 10000-99,999
wan = (thousand | hundred | suffix_ten | one_to_nine) + insert('万') + (thousand | (zero + hundred) | (delete('0') + zero + suffix_ten) | (delete('0') ** 2 + zero + one_to_nine) | delete('0') ** 4)
wan = wan.optimize()

#
yi = (wan | thousand | hundred | suffix_ten | one_to_nine) + insert('亿') + (thousand + insert('万') + (thousand | (zero + hundred) | (delete('0') + zero + suffix_ten) | (delete('0') ** 2 + zero + one_to_nine) | delete('0') ** 4) |
                                                                              (zero + hundred + insert('万') + (thousand | (zero + hundred) | (delete('0') + zero + suffix_ten) | (delete('0') ** 2 + zero + one_to_nine) | delete('0') ** 4)) |
                                                                              (delete('0') + zero + suffix_ten + insert('万') + (thousand | (zero + hundred) | (delete('0') + zero + suffix_ten) | (delete('0') ** 2 + zero + one_to_nine) | delete('0') ** 4)) |
                                                                              (delete('0') ** 2 + zero + one_to_nine + insert('万') + (thousand | (zero + hundred) | (delete('0') + zero + suffix_ten) | (delete('0') ** 2 + zero + one_to_nine) | delete('0') ** 4)) |
                                                                              (delete('0') ** 3 + zero + thousand) |
                                                                              (delete('0') ** 4 + zero + hundred) |
                                                                              (delete('0') ** 5 + zero + suffix_ten) |
                                                                              (delete('0') ** 6 + zero + one_to_nine) |
                                                                              delete('0') ** 8)
yi = yi.optimize()

fraction = pynini.cross(".", "点") + (zero | one_to_nine).plus
fraction = fraction.optimize()

for i in range(wan.num_states()):
    if wan.final(i) == pynini.Weight.one(wan.weight_type()):
        wan.set_final(i, 10)

for i in range(thousand.num_states()):
    if thousand.final(i) == pynini.Weight.one(thousand.weight_type()):
        thousand.set_final(i, 20)

for i in range(hundred.num_states()):
    if hundred.final(i) == pynini.Weight.one(hundred.weight_type()):
        hundred.set_final(i, 30)

for i in range(ten.num_states()):
    if ten.final(i) == pynini.Weight.one(ten.weight_type()):
        ten.set_final(i, 40)

for i in range(one_to_nine.num_states()):
    if one_to_nine.final(i) == pynini.Weight.one(one_to_nine.weight_type()):
        one_to_nine.set_final(i, 50)

number = pynini.cross("-", "负").ques + (yi | wan | thousand | hundred | ten | one_to_nine | zero)
number = (number + fraction.ques).optimize()
rule = pynini.cdrewrite(number, "", "", sigma)

for text in [ '4321', '4320', '4021', '4020', '4001', '4000',
        '54321', '54320', '54301', '54300', '54021', '54020', '54001', '54000', '50321', '50320', '50301',
        '50300', '50021', '50001', '50000',
        '100000000', '100000009', '100000080', '100000089', '100000700', '100000709', '100000780', '100000789',
        '100006000', '100006009', '100006080', '100006089', '100006700', '100006709', '100006780', '100006789',
        '100050000', '100050009', '100050080', '100050089', '100050700', '100050709', '100050780', '100050789',
        '100056000', '100056009', '100056080', '100056089', '100056700', '100056709', '100056780', '100056789',
        '100400000', '100400009', '100400080', '100400089', '100400700', '100400709', '100400780', '100400789',
        '100406000', '100406009', '100406080', '100406089', '100406700', '100406709', '100406780', '100406789',
        '100450000', '100450009', '100450080', '100450089', '100450700', '100450709', '100450780', '100450789',
        '100456000', '100456009', '100456080', '100456089', '100456700', '100456709', '100456780', '100456789',
        '103000000', '103000009', '103000080', '103000089', '103000700', '103000709', '103000780', '103000789',
        '103006000', '103006009', '103006080', '103006089', '103006700', '103006709', '103006780', '103006789',
        '103050000', '103050009', '103050080', '103050089', '103050700', '103050709', '103050780', '103050789',
        '103056000', '103056009', '103056080', '103056089', '103056700', '103056709', '103056780', '103056789',
        '103400000', '103400009', '103400080', '103400089', '103400700', '103400709', '103400780', '103400789',
        '103406000', '103406009', '103406080', '103406089', '103406700', '103406709', '103406780', '103406789',
        '103450000', '103450009', '103450080', '103450089', '103450700', '103450709', '103450780', '103450789',
        '103456000', '103456009', '103456080', '103456089', '103456700', '103456709', '103456780', '103456789',
        '120000000', '120000009', '120000080', '120000089', '120000700', '120000709', '120000780', '120000789',
        '120006000', '120006009', '120006080', '120006089', '120006700', '120006709', '120006780', '120006789',
        '120050000', '120050009', '120050080', '120050089', '120050700', '120050709', '120050780', '120050789',
        '120056000', '120056009', '120056080', '120056089', '120056700', '120056709', '120056780', '120056789',
        '120400000', '120400009', '120400080', '120400089', '120400700', '120400709', '120400780', '120400789',
        '120406000', '120406009', '120406080', '120406089', '120406700', '120406709', '120406780', '120406789',
        '120450000', '120450009', '120450080', '120450089', '120450700', '120450709', '120450780', '120450789',
        '120456000', '120456009', '120456080', '120456089', '120456700', '120456709', '120456780', '120456789',
        '123000000', '123000009', '123000080', '123000089', '123000700', '123000709', '123000780', '123000789',
        '123006000', '123006009', '123006080', '123006089', '123006700', '123006709', '123006780', '123006789',
        '123050000', '123050009', '123050080', '123050089', '123050700', '123050709', '123050780', '123050789',
        '123056000', '123056009', '123056080', '123056089', '123056700', '123056709', '123056780', '123056789',
        '123400000', '123400009', '123400080', '123400089', '123400700', '123400709', '123400780', '123400789',
        '123406000', '123406009', '123406080', '123406089', '123406700', '123406709', '123406780', '123406789',
        '123450000', '123450009', '123450080', '123450089', '123450700', '123450709', '123450780', '123450789',
        '123456000', '123456009', '123456080', '123456089', '123456700', '123456709', '123456780', '123456789',
        '-1', '1.3', "-1.3", "23.2", ".348", '-123456000', '8765432123456789', '-432123456789', '-432123456789.01'
]:
   out_put = list(pynini.shortestpath(pynini.compose(text, rule)).paths().ostrings())
   print(out_put)

rule.write('number.fst')

! ls -lh number.fst

['四千三百二十一']
['四千三百二十']
['四千零二十一']
['四千零二十']
['四千零一']
['四千']
['五万四千三百二十一']
['五万四千三百二十']
['五万四千三百零一']
['五万四千三百']
['五万四千零二十一']
['五万四千零二十']
['五万四千零一']
['五万四千']
['五万零三百二十一']
['五万零三百二十']
['五万零三百零一']
['五万零三百']
['五万零二十一']
['五万零一']
['五万']
['一亿']
['一亿零九']
['一亿零八十']
['一亿零八十九']
['一亿零七百']
['一亿零七百零九']
['一亿零七百八十']
['一亿零七百八十九']
['一亿零六千']
['一亿零六千零九']
['一亿零六千零八十']
['一亿零六千零八十九']
['一亿零六千七百']
['一亿零六千七百零九']
['一亿零六千七百八十']
['一亿零六千七百八十九']
['一亿零五万']
['一亿零五万零九']
['一亿零五万零八十']
['一亿零五万零八十九']
['一亿零五万零七百']
['一亿零五万零七百零九']
['一亿零五万零七百八十']
['一亿零五万零七百八十九']
['一亿零五万六千']
['一亿零五万六千零九']
['一亿零五万六千零八十']
['一亿零五万六千零八十九']
['一亿零五万六千七百']
['一亿零五万六千七百零九']
['一亿零五万六千七百八十']
['一亿零五万六千七百八十九']
['一亿零四十万']
['一亿零四十万零九']
['一亿零四十万零八十']
['一亿零四十万零八十九']
['一亿零四十万零七百']
['一亿零四十万零七百零九']
['一亿零四十万零七百八十']
['一亿零四十万零七百八十九']
['一亿零四十万六千']
['一亿零四十万六千零九']
['一亿零四十万六千零八十']
['一亿零四十万六千零八十九']
['一亿零四十万六千七百']
['一亿零四十万六千七百零九']
['一亿零四十万六千七百八十']
['一亿零四十万六千七百八十九']
['一亿零四十五万']
['一亿零四十五万零九']
['一亿零四十五万零八十']
['一亿零四十五万零八十九']
['一亿零四十五万零七百']
['一亿零四十五万零七百零九']
['一亿零四十五万零七百八十']
['一亿